In [199]:
import pandas as pd

In [200]:
df = pd.read_csv("To_clean.csv") 

In [201]:
df.head()

,ID Number,Submission Date,Application Property,Application City/Town,Application St Addresss,Application Unit,Source,Source New,Age,Race/Ethnicity,Disability,Current Residence,HH Size,Dependents,HH Type,HH Income,HH Assets,FTHB Class?
0,314921,10/21/23 7:05 PM,"Groton, 503 Main",Groton,503 Main St,NaN,MyMassHome,MyMassHome,44.0,Choose not to answer,No,Watertown,2,0,Single person,"60,000.00",2.00,Yes
1,827042,10/24/23 8:41 PM,North Andover Unit 109,North Andover,NaN,Unit 109,MyMassHome,MyMassHome,32.0,White,No,"Haverhill, MA",1,0,Single person,"62,608.00","26,866.69",Yes
2,152270,10/25/23 7:55 PM,North Andover Unit 109,North Andover,NaN,Unit 109,Zillow/Trulia/Etc.,Zillow/Trulia/Other Website,36.0,White,No,"Winthrop, MA",1,0,Single person,"58,000.00","25,000.00",No
3,949729,10/25/23 8:33 PM,North Andover Unit 109,North Andover,NaN,Unit 109,Zillow/Trulia/Etc.,Zillow/Trulia/Other Website,27.0,White,No,"Georgetown, MA",1,0,Single person,"62,000.00","62,000.00",No
4,814571,10/26/23 8:11 PM,North Andover Unit 109,North Andover,NaN,Unit 109,Zillow/Trulia/Etc.,Zillow/Trulia/Other Website,30.0,White,No,Haverhill,1,0,Single person,"60,000.00","33,034.00",No


In [202]:
df.shape

(1360, 18)

In [203]:
df.describe()

,ID Number,Age,HH Size,Dependents
count,1360.000000,1359.000000,1360.000000,1360.000000
mean,508826.384559,41.019500,2.175735,0.628676
std,285702.071923,13.606741,1.578227,0.922884
min,101.000000,0.000000,0.000000,0.000000
25%,251857.000000,31.000000,1.000000,0.000000
50%,520769.500000,38.000000,2.000000,0.000000
75%,744130.000000,49.000000,3.000000,1.000000
max,998583.000000,86.000000,31.000000,7.000000


In [204]:
# --- 1. Check missing values for each column ---
missing_summary = df.isnull().sum().to_frame(name='Missing_Values')
missing_summary['Missing_%'] = (missing_summary['Missing_Values'] / len(df) * 100).round(2)

# --- 2. Check number of unique values for each column ---
unique_summary = df.nunique().to_frame(name='Unique_Values')

# --- 3. Combine summaries ---
summary = missing_summary.join(unique_summary)

# --- 4. Display summary ---
print(summary)

                         Missing_Values  Missing_%  Unique_Values
ID Number                             0       0.00            965
Submission Date                       0       0.00           1312
Application Property                  1       0.07             69
Application City/Town                 1       0.07             46
Application St Addresss              26       1.91             69
Application Unit                    768      56.47             45
Source                                4       0.29             44
Source New                           24       1.76              6
Age                                   1       0.07             68
Race/Ethnicity                        3       0.22             63
Disability                            2       0.15              4
Current Residence                     1       0.07            397
HH Size                               0       0.00             10
Dependents                            0       0.00              7
HH Type   

In [205]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1360 entries, 0 to 1359
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   ID Number                1360 non-null   int64  
 1   Submission Date          1360 non-null   object 
 2   Application Property     1359 non-null   object 
 3   Application City/Town    1359 non-null   object 
 4   Application St Addresss  1334 non-null   object 
 5   Application Unit         592 non-null    object 
 6   Source                   1356 non-null   object 
 7   Source New               1336 non-null   object 
 8   Age                      1359 non-null   float64
 9   Race/Ethnicity           1357 non-null   object 
 10  Disability               1358 non-null   object 
 11  Current Residence        1359 non-null   object 
 12  HH Size                  1360 non-null   int64  
 13  Dependents               1360 non-null   int64  
 14  HH Type                 

In [206]:
df.columns

Index(['ID Number', 'Submission Date', 'Application Property',
       'Application City/Town', 'Application St Addresss', 'Application Unit',
       'Source', 'Source New', 'Age', 'Race/Ethnicity', 'Disability',
       'Current Residence', 'HH Size', 'Dependents', 'HH Type', 'HH Income',
       'HH Assets', 'FTHB Class?'],
      dtype='object')

In [207]:
del df['Source']

In [208]:
print(f"Unique current_residence formats (sample): {len(df['Current Residence'].unique())}")
print("Rows with comma:", df['Current Residence'].str.contains(",", na=False).sum())

Unique current_residence formats (sample): 398
Rows with comma: 111


In [209]:
# function to extract city and state from the 'Current Residence' column
import re
def extract_city_state(entry):
    if pd.isna(entry):
        return pd.Series([None, None])
    
    entry = entry.strip()

    # Try to split by comma
    if ',' in entry:
        parts = entry.split(',')
        city = parts[0].strip()
        state = parts[1].strip() if len(parts) > 1 else 'MA'
        return pd.Series([city, state])

    # Try to match City State pattern (e.g., "Springfield MA")
    match = re.match(r'([A-Za-z\s]+)\s+([A-Z]{2})$', entry)
    if match:
        city = match.group(1).strip()
        state = match.group(2).strip()
        return pd.Series([city, state])
    
    # If only city is present, assume MA
    return pd.Series([entry.title(), 'MA'])


In [210]:
# applying the function to the 'Current Residence' column

df[['Current Residence City', 'Current Residence State']] = df['Current Residence'].apply(extract_city_state)

# Preview results
print(df[['Current Residence', 'Current Residence City', 'Current Residence State']].head())
# Check for any remaining rows with missing city or state
missing_city_state = df[df['Current Residence City'].isnull() | df['Current Residence State'].isnull()]
print(f"Rows with missing city or state: {len(missing_city_state)}")

#print the row with the missin city or state
print(missing_city_state)

# Save cleaned data to a new CSV file
#df.to_csv("To_clean_cleaned.csv", index=False)

  Current Residence Current Residence City Current Residence State
0         Watertown              Watertown                      MA
1     Haverhill, MA              Haverhill                      MA
2      Winthrop, MA               Winthrop                      MA
3    Georgetown, MA             Georgetown                      MA
4         Haverhill              Haverhill                      MA
Rows with missing city or state: 1
    ID Number   Submission Date           Application Property  \
49      97210  1/10/24 12:00 AM  13 Coppersmith Way ~ Townsend   

   Application City/Town Application St Addresss Application Unit Source New  \
49              Townsend     13 Coppersmith Way               NaN        NaN   

    Age Race/Ethnicity Disability Current Residence  HH Size  Dependents  \
49  NaN            NaN         no               NaN        2           0   

          HH Type HH Income HH Assets FTHB Class? Current Residence City  \
49  two person HH       NaN       NaN   

In [211]:
#change submission date/time to separate columns 
# Step 1: Convert the string to a pandas datetime object
df['Submission Date'] = pd.to_datetime(df['Submission Date'])
# Step 2: Extract the date and time components
df['submission_date'] = df['Submission Date'].dt.date
df['submission_time'] = df['Submission Date'].dt.time

# Preview the result
print(df[['Submission Date', 'submission_date', 'submission_time']].head())

      Submission Date submission_date submission_time
0 2023-10-21 19:05:00      2023-10-21        19:05:00
1 2023-10-24 20:41:00      2023-10-24        20:41:00
2 2023-10-25 19:55:00      2023-10-25        19:55:00
3 2023-10-25 20:33:00      2023-10-25        20:33:00
4 2023-10-26 20:11:00      2023-10-26        20:11:00


/var/folders/g4/7ndpmb892dqg4slpc87nl3d00000gn/T/ipykernel_67034/514149535.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Submission Date'] = pd.to_datetime(df['Submission Date'])


In [212]:
# Standardizing the race/ethnicity column
df['Race/Ethnicity'].describe()

count      1357
unique       63
top       White
freq        663
Name: Race/Ethnicity, dtype: object

In [213]:
df['Race/Ethnicity'].value_counts().head(10)

Race/Ethnicity
White                                            663
Black or African American                        184
Hispanic or Latino                               122
Asian                                            122
Choose not to answer                              50
Asian/Pacific Islander                            43
Middle Eastern or North African                   26
Black or African American\nHispanic or Latino     15
Hispanic or Latino\nWhite                         13
Black or African American Hispanic/Latino         11
Name: count, dtype: int64

In [214]:
# Define lowercased canonical race labels
canonical_races = [
    "Asian",
    "Black or African American",
    "Hispanic or Latino",
    "Native American or Alaskan Native",
    "Middle Eastern or North African",
    "Pacific Islander or Native Hawaiian",
    "White",
    "Choose not to answer",
    "Other"
]

# Custom aliases (lowercased): free-text → canonical label
race_aliases = {
    "hispaniclatino": "Hispanic or Latino",
    "hispanic latino": "Hispanic or Latino",
    "hispanic/latino": "Hispanic or Latino",
    "latino": "Hispanic or Latino",
    "latin american": "Hispanic or Latino",
    "pacific islander": "Pacific Islander or Native Hawaiian",
    "native hawaiian": "Pacific Islander or Native Hawaiian",
    "black": "Black or African American",
    "african american": "Black or African American",
    "navajo": "Native American or Alaskan Native",
    "native american": "Native American or Alaskan Native",
    "cherokee": "Native American or Alaskan Native",
    "north african": "Middle Eastern or North African",
    "middle eastern": "Middle Eastern or North African",
    "egyptian": "Middle Eastern or North African",
    "afghan": "Asian",
    "brazilian": "Hispanic or Latino",
    "italian": "White",
    "greek": "White"
    # add more as needed...
}

canonical_races_lower = [r.lower() for r in canonical_races]

# Step 2: Function to extract matched races and unmatched free text
def extract_race_with_logging(text):
    if pd.isna(text):
        return pd.Series([[], None])

    text = text.lower()
    matched = []
    text_copy = text  # we’ll remove matched terms from this progressively

    for label, label_lc in zip(canonical_races, canonical_races_lower):
        # Regex: match exact label with word boundaries
        pattern = r'\b' + re.escape(label_lc) + r'\b'
        if re.search(pattern, text_copy):
            matched.append(label)
            text_copy = re.sub(pattern, '', text_copy)  # remove the matched part

    # 2. Match aliases using substring (not strict regex for flexibility)
    for alias, canonical in race_aliases.items():
        if alias in text_copy and canonical not in matched:
            matched.append(canonical)
            text_copy = text_copy.replace(alias, '')
            
    # After removing all known labels, clean up the rest to find unmatched text
    leftover = re.sub(r'[^a-zA-Z ]+', '', text_copy)  # remove punctuation/numbers
    leftover = re.sub(r'\s+', ' ', leftover).strip()  # normalize spaces

    return pd.Series([matched, leftover if leftover else None])



In [215]:
# Apply the function to dataset 
df[['race_list_chapa', 'unmatched_text']] = df['Race/Ethnicity'].apply(extract_race_with_logging)

#categorize the responses 
def classify_race(x):
    if len(x) == 0:
        return 'Missing'
    elif len(x) == 1:
        return 'Single'
    else:
        return 'Multiple'
    
df['race_response_type'] = df['race_list_chapa'].apply(classify_race)



In [216]:
# View entries with unmatched text
unmatched_df = df[df['unmatched_text'].notna()]
print(unmatched_df[['Race/Ethnicity', 'race_list_chapa', 'unmatched_text']].head())

# Save if needed
unmatched_df.to_csv("unmatched_race_entries.csv", index=False)

                      Race/Ethnicity              race_list_chapa  \
62          Hispanic Hispanic/Latino         [Hispanic or Latino]   
101           White, Lebanese/syrian                      [White]   
102    White, Latino Hispanic/Latino  [White, Hispanic or Latino]   
111  Hispanic Latino Hispanic/Latino         [Hispanic or Latino]   
120        Brazilian Hispanic/Latino         [Hispanic or Latino]   

     unmatched_text  
62         hispanic  
101  lebanesesyrian  
102          latino  
111  hispaniclatino  
120       brazilian  


In [217]:
# Calculate the number of races in each entry
df['num_races_selected'] = df['race_list_chapa'].apply(len)

# Get the maximum
max_races_selected = df['num_races_selected'].max()
print(max_races_selected)

4


In [218]:
df['race_list_string'] = df['race_list_chapa'].apply(lambda x: ', '.join(x) if isinstance(x, list) else '')


In [219]:
df

,ID Number,Submission Date,Application Property,Application City/Town,Application St Addresss,Application Unit,Source New,Age,Race/Ethnicity,Disability,...,FTHB Class?,Current Residence City,Current Residence State,submission_date,submission_time,race_list_chapa,unmatched_text,race_response_type,num_races_selected,race_list_string
0,314921,2023-10-21 19:05:00,"Groton, 503 Main",Groton,503 Main St,NaN,MyMassHome,44.0,Choose not to answer,No,...,Yes,Watertown,MA,2023-10-21,19:05:00,[Choose not to answer],None,Single,1,Choose not to answer
1,827042,2023-10-24 20:41:00,North Andover Unit 109,North Andover,NaN,Unit 109,MyMassHome,32.0,White,No,...,Yes,Haverhill,MA,2023-10-24,20:41:00,[White],None,Single,1,White
2,152270,2023-10-25 19:55:00,North Andover Unit 109,North Andover,NaN,Unit 109,Zillow/Trulia/Other Website,36.0,White,No,...,No,Winthrop,MA,2023-10-25,19:55:00,[White],None,Single,1,White
3,949729,2023-10-25 20:33:00,North Andover Unit 109,North Andover,NaN,Unit 109,Zillow/Trulia/Other Website,27.0,White,No,...,No,Georgetown,MA,2023-10-25,20:33:00,[White],None,Single,1,White
4,814571,2023-10-26 20:11:00,North Andover Unit 109,North Andover,NaN,Unit 109,Zillow/Trulia/Other Website,30.0,White,No,...,No,Haverhill,MA,2023-10-26,20:11:00,[White],None,Single,1,White
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1355,583613,2025-01-14 17:05:55,72 Buttercup Lane ~ Grafton,Grafton,72 Buttercup Lane,NaN,Real Estate Agent,36.0,Middle Eastern or North African,No,...,No,Worcester,MA,2025-01-14,17:05:55,[Middle Eastern or North African],None,Single,1,Middle Eastern or North African
1356,575544,2025-01-12 15:11:57,72 Buttercup Lane ~ Grafton,Grafton,72 Buttercup Lane,NaN,Zillow/Trulia/Other Website,25.0,White,No,...,No,Sutton,MA,2025-01-12,15:11:57,[White],None,Single,1,White
1357,954465,2025-01-11 23:11:20,72 Buttercup Lane ~ Grafton,Grafton,72 Buttercup Lane,NaN,MyMassHome,62.0,White,No,...,No,Northborough,MA,2025-01-11,23:11:20,[White],None,Single,1,White
1358,218300,2025-01-02 13:32:05,"250 Main St, Unit 410 ~ Hudson (age restricted...",Hudson,250 Main St,Unit 410,Zillow/Trulia/Other Website,60.0,White,Yes,...,No,Hudson,MA,2025-01-02,13:32:05,[White],None,Single,1,White


In [220]:
df.to_csv("To_clean_cleaned.csv", index=False)

In [221]:
# Step 1: Normalize order by splitting and sorting
def normalize_combination(race_string):
    if pd.isna(race_string) or race_string.strip() == '':
        return 'Missing'
    parts = [part.strip() for part in race_string.split(',')]
    parts_sorted = sorted(parts)
    return ', '.join(parts_sorted)

# Step 2: Apply normalization to a new column
df['race_combination_normalized'] = df['race_list_string'].apply(normalize_combination)

# Step 3: Count unique combinations (ignoring order)
combination_counts = df['race_combination_normalized'].value_counts()

# View result
print(combination_counts)
len(df['race_combination_normalized'].unique())

race_combination_normalized
White                                                                           667
Black or African American                                                       185
Hispanic or Latino                                                              143
Asian                                                                           128
Choose not to answer                                                             51
Asian, Pacific Islander or Native Hawaiian                                       44
Middle Eastern or North African                                                  33
Black or African American, Hispanic or Latino                                    26
Hispanic or Latino, White                                                        23
Black or African American, White                                                 10
Missing                                                                          10
Middle Eastern or North African, White          

27

In [222]:
df.loc[df['race_combination_normalized'] == 'Choose not to answer, Hispanic or Latino', 'race_combination_normalized'] = 'Hispanic or Latino'
df['race_combination_normalized'].value_counts()

race_combination_normalized
White                                                                           667
Black or African American                                                       185
Hispanic or Latino                                                              149
Asian                                                                           128
Choose not to answer                                                             51
Asian, Pacific Islander or Native Hawaiian                                       44
Middle Eastern or North African                                                  33
Black or African American, Hispanic or Latino                                    26
Hispanic or Latino, White                                                        23
Missing                                                                          10
Black or African American, White                                                 10
Middle Eastern or North African, White          

In [223]:
df.loc[df['race_combination_normalized'] == 'Missing', 
       ['Race/Ethnicity', 'race_list_chapa', 'unmatched_text', 'race_response_type', 'race_list_string', 'race_combination_normalized']]


,Race/Ethnicity,race_list_chapa,unmatched_text,race_response_type,race_list_string,race_combination_normalized
49,NaN,[],None,Missing,,Missing
127,Iraqi-American,[],iraqiamerican,Missing,,Missing
155,NaN,[],None,Missing,,Missing
234,NaN,[],None,Missing,,Missing
562,South Asia - Pakistan,[],south asia pakistan,Missing,,Missing
581,cape Verde Portuguese,[],cape verde portuguese,Missing,,Missing
582,European/African,[],europeanafrican,Missing,,Missing
653,Uyghur,[],uyghur,Missing,,Missing
723,Cape Verdean,[],cape verdean,Missing,,Missing
735,Pakistani & Ukrainian,[],pakistani ukrainian,Missing,,Missing


In [224]:
# Manually changing race normalized values 
df.loc[df['unmatched_text'] == 'iraqiamerican', 'race_combination_normalized'] = 'Middle Eastern or North African'
df.loc[df['unmatched_text'] == 'south asia pakistan', 'race_combination_normalized'] = 'Asian'
df.loc[df['unmatched_text'] == 'europeanafrican', 'race_combination_normalized'] = 'Black or African American, White'
df.loc[df['unmatched_text'] == 'cape verde portuguese', 'race_combination_normalized'] = 'Black or African American'
df.loc[df['unmatched_text'] == 'uyghur', 'race_combination_normalized'] = 'Asian'
df.loc[df['unmatched_text'] == 'cape verdean', 'race_combination_normalized'] = 'Black or African American'
df.loc[df['unmatched_text'] == 'pakistani ukrainian', 'race_combination_normalized'] = 'Asian, White'

df.loc[df['race_combination_normalized'] == 'Missing', 
       ['Race/Ethnicity', 'race_list_chapa', 'unmatched_text', 'race_response_type', 'race_list_string', 'race_combination_normalized']]

,Race/Ethnicity,race_list_chapa,unmatched_text,race_response_type,race_list_string,race_combination_normalized
49,NaN,[],None,Missing,,Missing
155,NaN,[],None,Missing,,Missing
234,NaN,[],None,Missing,,Missing


In [225]:
df.to_csv("To_clean_cleaned.csv", index=False)

In [226]:
df['race_combination_normalized'].unique()

array(['Choose not to answer', 'White', 'Black or African American',
       'Asian, Pacific Islander or Native Hawaiian', 'Hispanic or Latino',
       'Missing', 'Black or African American, Hispanic or Latino',
       'Asian, Black or African American, Pacific Islander or Native Hawaiian, White',
       'Middle Eastern or North African',
       'Asian, Pacific Islander or Native Hawaiian, White',
       'Hispanic or Latino, White', 'Black or African American, White',
       'Black or African American, Native American or Alaskan Native, White',
       'Asian',
       'Black or African American, Native American or Alaskan Native',
       'Asian, Black or African American, Middle Eastern or North African, White',
       'Middle Eastern or North African, White', 'Asian, White',
       'Native American or Alaskan Native', 'Other',
       'Hispanic or Latino, Native American or Alaskan Native',
       'Native American or Alaskan Native, White',
       'Asian, Middle Eastern or North African,

In [227]:
# Census Race Categories

# Custom aliases (lowercased): free-text → canonical label
census_race_aliases = {
    'Choose not to answer': 'Missing',
    'White': 'White alone',
    'Black or African American': 'Black or African American alone',
    'Asian, Pacific Islander or Native Hawaiian': 'Asian; Native Hawaiian or Other Pacific Islander',
    'Hispanic or Latino': 'Missing',
    'Missing': 'Missing',
    'Black or African American, Hispanic or Latino': 'Black or African American',
    'Asian, Black or African American, Pacific Islander or Native Hawaiian, White':'Asian, Black or African American, Native Hawaiian or Other Pacific Islander, White',
    'Middle Eastern or North African': 'White alone',
    'Asian, Pacific Islander or Native Hawaiian, White': 'Asian; Native Hawaiian or Other Pacific Islander; White',
    'Hispanic or Latino, White': 'White alone',
    'Black or African American, White': 'Black or African American; White',
    'Black or African American, Native American or Alaskan Native, White': 'Black or African American; American Indian and Alaska Native; White',
    'Asian': 'Asian alone',
    'Black or African American, Native American or Alaskan Native': 'Black or African American; American Indian and Alaska Native',
    'Asian, Black or African American, Middle Eastern or North African, White': 'Asian; Black or African American; White',
    'Middle Eastern or North African, White': 'White alone',
    'Asian, White': 'Asian; White',
    'Native American or Alaskan Native': 'American Indian and Alaska Native alone',
    'Other': 'Some Other Race',
    'Hispanic or Latino, Native American or Alaskan Native': 'Hispanic or Latino; American Indian and Alaska Native',
    'Native American or Alaskan Native, White': 'American Indian and Alaska Native; White',
    'Asian, Middle Eastern or North African, White': 'Asian; White',
    'Asian, Middle Eastern or North African': 'Asian; White',
    'Black or African American, Hispanic or Latino, White': 'Black or African American; White',
    'Pacific Islander or Native Hawaiian': 'Native Hawaiian or Other Pacific Islander alone',
}

# Step 1: Define the function
def map_to_census_category(race_combo):
    # Use dictionary lookup, return 'Unmapped' if no match found
    return census_race_aliases.get(race_combo, 'Unmapped')

# Step 2: Apply it to your DataFrame
df['census_races'] = df['race_combination_normalized'].apply(map_to_census_category)


In [228]:
df.to_csv("To_clean_cleaned.csv", index=False)

In [229]:
# Houshold Type cleaning 
df['HH Type'].unique()

array(['Single person', 'Single parent',
       'Married/partners/couple, with dependents',
       'Married/partners/couple, no dependents',
       'More than one related adults, with dependents', 'two person HH',
       'Currently in divorce process',
       'More than one related adults, no dependents', 'one brother',
       'married', 'Domestic Partners 1 child 13yrs', 'separated',
       'Married',
       'Divorced with More than one related adults, with dependents',
       'DIVORCED WITH More than one related adults, with dependents',
       'Live with parents', 'Separated with 1 dependent',
       'Single, would like to foster/adopt', 'Married - separated',
       'Siblings', 'myself and grandmother', 'Parents/sisters',
       'Single/ mothers caregiver',
       'Brother and Sister (both adults, require 2 bedrooms)',
       'Married couple and one adult', 'Partners',
       'engaged with a child 16 months old', 'Living with parents',
       'Separated, with dependents',
       'S

In [234]:
canonical_hh_types = [
    "Single person",
    "Married/partners/couple, no dependents",
    "Married/partners/couple, with dependents",
    "Single parent",
    "Other", 
    "More than one related adults, with dependents",
    "More than one related adults, no dependents"
]

hh_aliases = {
    "single person": "Single person",
    "living alone": "Single person",
    "solo": "Single person",
    "separated": "Single person",
    "siblings": "More than one related adults, no dependents",
    
    "more than one related adults, with dependents": "More than one related adults, with dependents",
    "more than one related adults, no dependents": "More than one related adults, with dependents",
    
    "married": "Married/partners/couple, no dependents",  # base case
    "married no kids": "Married/partners/couple, no dependents",
    "couple no children": "Married/partners/couple, no dependents",
    "partnered no dependents": "Married/partners/couple, no dependents",

    "married with kids": "Married/partners/couple, with dependents",
    "couple with dependents": "Married/partners/couple, with dependents",
    "married children": "Married/partners/couple, with dependents",
    "partnered with kids": "Married/partners/couple, with dependents",

    "single mom": "Single parent",
    "single dad": "Single parent",
    "single parent": "Single parent",
    "solo parent": "Single parent",

    "other": "Other"
    # add more free-text variants as needed
}

def extract_hh_type(text):
    if pd.isna(text):
        return pd.Series([None, None])
    
    text = text.lower().strip()

    matched = None
    for alias, canonical in hh_aliases.items():
        if alias in text:
            matched = canonical
            break

    leftover = None if matched else text  # keep unmatched as leftover
    return pd.Series([matched, leftover])

# Apply the function to the 'HH Type' column
df[['hh_type_standardized', 'hh_unmatched']] = df['HH Type'].apply(extract_hh_type)

# Preview unmatched free-text entries
df[df['hh_unmatched'].notna()][['HH Type', 'hh_type_standardized','hh_unmatched']]

,HH Type,hh_type_standardized,hh_unmatched
49,two person HH,None,two person hh
73,Currently in divorce process,None,currently in divorce process
157,Domestic Partners 1 child 13yrs,None,domestic partners 1 child 13yrs
288,Live with parents,None,live with parents
299,"Single, would like to foster/adopt",None,"single, would like to foster/adopt"
432,Parents/sisters,None,parents/sisters
506,Partners,None,partners
581,engaged with a child 16 months old,None,engaged with a child 16 months old
661,Living with parents,None,living with parents
758,"Single/living with parents, with dependents",None,"single/living with parents, with dependents"


In [238]:
# Manually changing HH Type  values 
df.loc[df['hh_unmatched'] == 'two person hh', 'hh_type_standardized'] = 'More than one related adults, no dependents'
df.loc[df['hh_unmatched'] == 'currently in divorce process', 'hh_type_standardized'] = 'Single person'
df.loc[df['hh_unmatched'] == 'domestic partners 1 child 13yrs', 'hh_type_standardized'] = 'Married/partners/couple, with dependents'
df.loc[df['hh_unmatched'] == 'live with parents', 'hh_type_standardized'] = 'More than one related adults, no dependents'
df.loc[df['hh_unmatched'] == 'single, would like to foster/adopt', 'hh_type_standardized'] = 'Single person'
df.loc[df['hh_unmatched'] == 'parents/sisters', 'hh_type_standardized'] = 'More than one related adults, no dependents'
df.loc[df['hh_unmatched'] == 'partners', 'hh_type_standardized'] = 'Married/partners/couple, no dependents'
df.loc[df['hh_unmatched'] == 'engaged with a child 16 months old', 'hh_type_standardized'] = 'Married/partners/couple, with dependents'
df.loc[df['hh_unmatched'] == 'living with parents', 'hh_type_standardized'] = 'More than one related adults, no dependents'
df.loc[df['hh_unmatched'] == 'single/living with parents, with dependents', 'hh_type_standardized'] = 'More than one related adults, with dependents'
df.loc[df['hh_unmatched'] == 'daughter and father', 'hh_type_standardized'] = 'Single parent'
df.loc[df['hh_unmatched'] == 'single, with live in aide', 'hh_type_standardized'] = 'Single person'
df.loc[df['hh_unmatched'] == 'divorced', 'hh_type_standardized'] = 'Single person'
df.loc[df['hh_unmatched'] == 'my aunt &uncle they live with me', 'hh_type_standardized'] = 'More than one related adults, no dependents'
df.loc[df['hh_unmatched'] == 'my aunt & uncle they live with me.', 'hh_type_standardized'] = 'More than one related adults, no dependents'


# Preview unmatched free-text entries
df[df['hh_type_standardized'].isna()][['HH Type', 'hh_type_standardized','hh_unmatched']]

,HH Type,hh_type_standardized,hh_unmatched


In [239]:
df.to_csv("To_clean_cleaned.csv", index=False)

In [245]:
df['HH Income'].min()


TypeError: '<=' not supported between instances of 'str' and 'float'